In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from pathlib import Path
import random


2026-09-04 16:55:55.248788: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-04 16:55:55.282980: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-04 16:55:56.183660: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# --- CONFIGURACIÓN DE RUTAS ---
BASE_DIR = Path("..")

# El modelo maestro
MODELS_DIR = BASE_DIR / "models"
MODEL_PATH = MODELS_DIR / "modelo_residuos_rpi_V3_05.keras"

# Tu carpeta con fotos reales tomadas por ti
IMAGES_TEST_DIR = BASE_DIR / "tests" / "images"

# Definimos las clases exactamente en el mismo orden alfabético que se entrenaron
class_names = ['crushed_metal', 'crushed_plastic', 'metal', 'no_reciclable', 'plastic']

print(f"Buscando imágenes de prueba en: {IMAGES_TEST_DIR.resolve()}")

Buscando imágenes de prueba en: /home/jesus/Documentos/GitHub/EntrenamientoIA/tests/images


In [3]:
print("Cargando el modelo...")
model = tf.keras.models.load_model(MODEL_PATH)
print("✅ Modelo cargado y listo para inferencia.")

Cargando el modelo...


I0000 00:00:1788562557.129081  427832 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 120 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1788562557.143113  427832 cuda_executor.cc:508] failed to allocate 120.12MiB (125960192 bytes) from device: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory


✅ Modelo cargado y listo para inferencia.


In [4]:
def analizar_residuo_real(ruta_imagen):
    # 1. Cargar imagen original solo para visualizarla bonita al final
    img_original = image.load_img(ruta_imagen)
    
    # 2. Cargar y preprocesar la imagen para el modelo
    img_preprocesada = image.load_img(ruta_imagen, target_size=(224, 224))
    img_array = image.img_to_array(img_preprocesada)

    
    # Keras espera un "lote" (batch) de imágenes, así que agregamos una dimensión extra
    img_array = np.expand_dims(img_array, axis=0) # Forma final: (1, 224, 224, 3)
    
    # 3. Predicción
    predicciones = model.predict(img_array, verbose=0) # verbose=0 para no saturar la consola
    indice_ganador = np.argmax(predicciones[0])
    porcentaje_confianza = predicciones[0][indice_ganador] * 100
    categoria_predicha = class_names[indice_ganador]
    
    # 4. Mostrar Resultados
    plt.figure(figsize=(6, 6))
    plt.imshow(img_original)
    
    # Si está muy seguro (>80%) el texto será verde, si duda, será naranja
    color_texto = 'green' if porcentaje_confianza > 80 else 'darkorange'
    
    plt.title(f"Predicción: {categoria_predicha.upper()}\nConfianza: {porcentaje_confianza:.2f}%", 
                fontsize=14, color=color_texto, fontweight='bold')
    plt.axis('off')
    plt.show()

In [6]:
# Verificamos si la carpeta existe
if IMAGES_TEST_DIR.exists():
    # Buscamos todas las imágenes JPG o PNG
    fotos_reales = [f for f in IMAGES_TEST_DIR.iterdir() if f.is_file() and f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    
    if fotos_reales:
        print(f"📸 ¡Se encontraron {len(fotos_reales)} fotos en la carpeta de pruebas!")
        
        # Elegimos hasta 6 fotos al azar para no saturar la pantalla
        cantidad_a_probar = min(6, len(fotos_reales))
        fotos_seleccionadas = random.sample(fotos_reales, cantidad_a_probar)
        
        for foto in fotos_seleccionadas:
            print(f"\n--- Analizando: {foto.name} ---")
            analizar_residuo_real(foto)
            
    else:
        print("La carpeta tests/images existe, pero está vacía o no tiene imágenes JPG/PNG.")
else:
    print(f"⚠️ No se encontró la ruta {IMAGES_TEST_DIR}.")
    print("Por favor, crea la carpeta y coloca algunas fotos tomadas con tu celular o la cámara web.")

📸 ¡Se encontraron 14 fotos en la carpeta de pruebas!

--- Analizando: pruebas (5).jpeg ---


E0000 00:00:1788562595.683302  427956 cuda_blas.cc:196] failed to create cublas handle: the resource allocation failed
E0000 00:00:1788562595.683320  427956 cuda_blas.cc:199] Failure to initialize cublas may be due to OOM (cublas needs some free memory when you initialize it, and your deep-learning framework may have preallocated more than its fair share), or may be because this binary was not built with support for the GPU in your machine.
E0000 00:00:1788562595.686245  427956 cuda_blas.cc:196] failed to create cublas handle: the resource allocation failed
E0000 00:00:1788562595.686253  427956 cuda_blas.cc:199] Failure to initialize cublas may be due to OOM (cublas needs some free memory when you initialize it, and your deep-learning framework may have preallocated more than its fair share), or may be because this binary was not built with support for the GPU in your machine.
2026-09-04 16:56:35.688529: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc)

UnknownError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/jesus/Documentos/GitHub/EntrenamientoIA/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/jesus/Documentos/GitHub/EntrenamientoIA/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/jesus/Documentos/GitHub/EntrenamientoIA/.venv/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/home/jesus/Documentos/GitHub/EntrenamientoIA/.venv/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/local/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/usr/local/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once

  File "/usr/local/lib/python3.12/asyncio/event